In [93]:
from bs4 import BeautifulSoup as soup
from bs4 import Tag, NavigableString
import mwparserfromhell
import requests
import pandas
import re

In [94]:
# Test code for downloaded file wiki-'eye'.html
# request page for eye online

with open("wiki-'eye'.html", 'r', encoding='utf-8') as f:
    page = f.read()
soupify = soup(page, 'html.parser')
# wikicode = mwparserfromhell.parse(soupify)
# mw-heading 2 for language, 
#   mw-heading 3 for subheadings like etymology, pronunciation, etc. (look at next element: paragraph, list, etc.)
    # span class="IPA nowrap" for pronunciation?
        # pronunciation can be mw-heading 4???
            # mayhaps id contains "Pronunciation"?, then look for list element
            # thus, id contains "Etymology" for etymology section?
    # span class="etyl" for etymology links (Ole English, Middle English)
    # i class="Latn" for Latin script (for non-English words), mention class in the etymology section
    # WHY ISN'T THE WIKIMEDIA PAGE NESTED AT ALL!!!?!?!!    
# wikicode.filter_templates()
soupify.find_all('h3') # seemingly gets relevant headings for this page

# feels a bit specific but worth a shot
soupify.find_all('h3')[0].parent.next_sibling.next_sibling.next_element.find('span', class_='IPA').text # get IPA pronuncation from first h3 heading (pronunciation) and next sibling (list of pronunciations)'

# okay, let's try parsing the etymology section, which is the second h3 heading
all_etyl = soupify.find_all('h3')[1].parent.next_sibling.next_sibling.find_all('i', class_='Latn')
[x.text for x in all_etyl]

# unfortunately, we cannot get the related term "ogle" at all - no ids or anything

['eye', 'yë', 'eyghe', 'ēage', '*augā', '*augô', '*h₃okʷ-', '*h₃ekʷ-']

In [109]:
def get_etym_links(etym_p): 
    # etym_links structure: {language: [{"word": word, "link": link}, ...], ...}
    etym_links = {"Other" : [], "Unknown": []}
    language = None
    for element in etym_p.descendants:
        # if element is a link to a language, add as key to dictionary and append any other etmyology links to its values
        if element.name == "span" and element.get("class") == ["etyl"]:
            language = element.text
            if language not in etym_links:
                etym_links[language] = []
        elif element.name == "i" and element.get("class") == ["Latn"]: # and language:
            link = element.next_sibling
            word_dict = {"word": element.text, "link" : link.get("href") if link and link.name == "a" else None}
            if language:
                etym_links[language].append(word_dict)
            else:
                etym_links["Unknown"].append(word_dict)
        elif element.name == "a" and language:
            word_dict = {"word": element.text, "link" : element.get("href")}
            etym_links["Other"].append(word_dict)
        
    return etym_links

In [96]:
def get_pronunciation(pron_ul): 
    # for li in pron_ul.find_all("li"):
        # if li.find("span", class_="IPA nowrap"):
        #     return li.find("span", class_="IPA nowrap").text
        return pron_ul.find("span", class_="IPA nowrap").text if pron_ul.find("span", class_="IPA nowrap") else None

In [97]:
def parse_etym_links(etym_links): 
    return

In [ ]:
# language_headings = page_soup.find_all("h2")
    # print(language_headings)
    # find english-language section (id="English") and remove all other language sections and their contents
    # language_headings are a result set, not a list, so we need to find the index of the relevant language section
    # index = 0
    # while index < len(language_headings):
    #     if language_headings[index].get('id') == language:
    #         break
    #     index += 1
    # print(type(language_headings[index]))
    # find next language section, remove it and all elements after it
    # get the actual html section for the relevant language
    language_section = page_soup.find("h2", id=language)
    # get the next heading 2 element, which is the next language section, if any exists
    # grab all the html following the language_section
    # print(language_section.find_all_next())
    relevant_html = language_section.find_all_next()
    # remove all html following the next language section, if it exists
    relevant_html = relevant_html[:relevant_html.index(language_section.find_next("h2"))] if language_section.find_next("h2") else relevant_html
    # next_section = language_section.find_next("h2")
    # relevant_html = language_section.find_all_next() - next_section.find_all_next() if next_section else language_section.find_all_next()
    # print(relevant_html)

In [107]:
def filter_language_sections(page_soup, language): 
    headers = page_soup.find_all("h2")
    # 
    language_section_idx = headers.index(page_soup.find("h2", id=language))
    language_section = headers[language_section_idx]

    relevant_html = []    
    for html in language_section.find_all_next():
        if html.name == "h2" and html != language_section:
            break
        if isinstance(html, Tag):
            relevant_html.append(html)    

    print("Language section: ", language_section)
    # print("Next language section: ", next_language)
    # for element in list(language_section.next_siblings):
    #     if element == next_language:
    #         break
    #     element.decompose()
    return soup("".join(str(html) for html in relevant_html), "html.parser")

In [110]:
# Scrapes Wiktionary page for relevant links, etymology, pronunciation, compacts into pandas df
def scrape_wiktionary_page(html_file):
    
    # url = f"https://en.wiktionary.org/wiki/{word}"
    # page = requests.get(url)
    print(html_file)
    if html_file.startswith("http"):
        grabbed_html = grab_page_html(html_file)
        html_content = grabbed_html#.decode("utf-8")
        word = html_file.split("/")[-1].split(".")[0] # get word from file name
    else:
        with open(html_file, "r", encoding="utf-8") as f:
            html_content = f.read()
        word = html_file.split(".")[0].split("-")[-1] # get word from file name
    page_soup = soup(html_content, "html.parser")

    # print(page_soup)
    relevant_html = filter_language_sections(page_soup, "English")
    # print("Relevant html: ", relevant_html)

    # grab all etymology sections that have id containing "Etymology" that are in the english-language section (some words have multiple etymologies)
    etymology = relevant_html.find_all("h3", id=re.compile("Etymology"))
    etym_dict = {}
    if etymology:
        for etym in etymology:
            if etym.find_next("p"):
                etym_dict[etym] = get_etym_links(etym.find_next("p"))

    # if etymology:
    #     for etym in etymology:
    #         if etym.find_next("p"):
    #             # grab all links in the etymology section
    #             get_etym_links(etym.find_next("p"))
    #             etymology_text = etym.find_next("p").text

    # else:
    #     etymology_text = None

    # Get pronunciation
    pronunciation = relevant_html.find("h3", id="Pronunciation")

    if pronunciation:
        pronunciation_ul = pronunciation.find_next("ul")
        pronunciation_text = get_pronunciation(pronunciation_ul)
    else:
        pronunciation_text = None

    # # Get related links (e.g., synonyms, antonyms, related terms)
    # related_links = []
    # for section in page_soup.find_all("h3", class_="mw-headline"):
    #     if section.text in ["Synonyms", "Antonyms", "Related terms" ]:
    #         links = section.find_next("ul").find_all("a")
    #         related_links.extend([link.text for link in links])

    # Compile into pandas DataFrame
    data = {
        "Word": word,
        "Etymology": etym_dict,
        "Pronunciation": pronunciation_text,
        # "Related Links": related_links
        # "Further Enquiry" : 
    }
    
    df = pandas.DataFrame([data])
    return df
    
def grab_page_html(word):
    url = f"https://en.wiktionary.org/wiki/{word}"
    page = requests.get(url)
    return page.content

In [112]:
df = scrape_wiktionary_page("wiki-'eye'.html")
df

wiki-'eye'.html
Language section:  <h2 id="English">English</h2>


,Word,Etymology,Pronunciation
0,'eye',{['Etymology 1']: {'Other': [{'word': 'Middle ...,/aɪ/


In [ ]:
# 200 ihops is one really good tattoo btw 
df["Etymology"][0]
# for item in df["Etymology"][0]:
#     print(item)

<h3 id="Etymology_1">Etymology 1</h3>
<h3 id="Etymology_2">Etymology 2</h3>
<h3 id="Etymology_3">Etymology 3</h3>
